# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [7]:
df = pd.read_csv("data/AviationData.csv", encoding="latin1", low_memory=False)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [6]:
df.info()
df.isna().sum().sort_values(ascending=False)
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [8]:
# View all columns
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [10]:
df[[
    "Event.Id",
    "Event.Date",
    "Country",
    "Aircraft.Category",
    "Aircraft.damage",
    "Make",
    "Model",
    "Number.of.Engines",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Weather.Condition"
]].head()

,Event.Id,Event.Date,Country,Aircraft.Category,Aircraft.damage,Make,Model,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition
0,20001218X45444,1948-10-24,United States,NaN,Destroyed,Stinson,108-3,1.0,2.0,0.0,UNK
1,20001218X45447,1962-07-19,United States,NaN,Destroyed,Piper,PA24-180,1.0,4.0,0.0,UNK
2,20061025X01555,1974-08-30,United States,NaN,Destroyed,Cessna,172M,1.0,3.0,NaN,IMC
3,20001218X45448,1977-06-19,United States,NaN,Destroyed,Rockwell,112,1.0,2.0,0.0,IMC
4,20041105X01764,1979-08-02,United States,NaN,Destroyed,Cessna,501,NaN,1.0,2.0,VMC


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [12]:
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

In [13]:
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

The client is interested in aircraft safety in terms of injury outcomes and structural damage.
Missing injury values are assumed to represent zero recorded injuries (no injuries reported in the incident).
Aircraft damage is used as a proxy for structural destruction, where “Destroyed” indicates total loss of the aircraft.
These assumptions are standard in aviation safety analysis and ensure consistent metric computation.

In [14]:
# Ensure injury columns are numeric
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [15]:
df[injury_cols] = df[injury_cols].fillna(0)

In [16]:
df["Aircraft.damage"] = df["Aircraft.damage"].fillna("Unknown")
df["Aircraft.damage"] = df["Aircraft.damage"].str.upper().str.strip()

In [17]:
df["Total_Injuries"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"] +
    df["Total.Minor.Injuries"]
)

In [18]:
df["Fatal_Event"] = df["Total.Fatal.Injuries"] > 0

In [19]:
df["Destroyed"] = df["Aircraft.damage"].str.contains("DESTROYED", na=False)

In [20]:
df["Serious_Injury_Event"] = df["Total.Serious.Injuries"] > 0

In [21]:
# Count incidents per aircraft model
model_counts = df.groupby(["Make", "Model"]).size().reset_index(name="Incident_Count")

# Merge back into dataset
df = df.merge(model_counts, on=["Make", "Model"], how="left")

# Keep only models with sufficient data (≥20 incidents)
df = df[df["Incident_Count"] >= 20]

The dataset was cleaned by converting injury-related columns to numeric format and imputing missing values as zero under the assumption of no recorded injuries. Aircraft damage was standardized to allow consistent identification of destroyed aircraft.
Several derived metrics were created, including total injuries per incident, binary indicators for fatal and serious injury events, and a destruction flag based on aircraft damage status.
To ensure statistical reliability, aircraft models with fewer than 20 recorded incidents were excluded from further analysis.

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

The dataset does not directly provide total passengers on board each flight.
Therefore, total occupants are estimated using the sum of all recorded injury categories and uninjured passengers.
This provides a reasonable proxy for total exposure per incident and allows calculation of injury likelihood as a proportion of occupants.

In [22]:
df["Total_Occupants"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"] +
    df["Total.Minor.Injuries"] +
    df["Total.Uninjured"]
)

In [25]:
df["Total_Occupants"] = df["Total_Occupants"].replace(0, np.nan)

In [26]:
df["Fatal_Injury_Rate"] = df["Total.Fatal.Injuries"] / df["Total_Occupants"]

In [27]:
df["Serious_Injury_Rate"] = df["Total.Serious.Injuries"] / df["Total_Occupants"]

In [28]:
df["Severe_Injury_Rate"] = (
    df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
) / df["Total_Occupants"]

In [29]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

Injury risk was normalized by estimating total occupants per flight using the sum of fatalities, serious injuries, minor injuries, and uninjured passengers.
This allows comparison across aircraft types regardless of flight size.
Fatal and serious injury rates were computed as proportions of total occupants, enabling standardized safety comparison between aircraft makes and models.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

The “Aircraft.damage” column contains categorical descriptions of aircraft structural condition after an incident.
Some values are missing or inconsistently formatted, so standardization is required.
We assume that missing values represent unknown damage status and should not be treated as “no damage.”

In [30]:
# Standardize text format
df["Aircraft.damage"] = df["Aircraft.damage"].str.upper().str.strip()

In [ ]:
df["Aircraft.damage"] = df["Aircraft.damage"].fillna("UNKNOWN")

In [31]:
df["Aircraft.damage"].value_counts()

Aircraft.damage
SUBSTANTIAL    39712
DESTROYED      11185
UNKNOWN         1215
MINOR           1093
Name: count, dtype: int64

In [32]:
df["Destroyed"] = df["Aircraft.damage"].str.contains("DESTROYED", na=False)

In [33]:
df["Destroyed"] = df["Destroyed"].astype(int)

In [34]:
df["Destroyed"].value_counts()

Destroyed
0    42020
1    11185
Name: count, dtype: int64

A new binary variable “Destroyed” was created from the Aircraft.damage field to capture whether an aircraft was completely destroyed in an accident.
This transformation allows quantification of structural failure rates across aircraft makes and models.
The variable is binary (1 = destroyed, 0 = not destroyed), enabling aggregation into destruction rates for comparative safety analysis.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

The “Make” column contains aircraft manufacturer names that are inconsistently formatted and include rare manufacturers with very few observations.
To ensure statistical reliability and meaningful comparisons, the following cleaning steps are applied:

Standardize text format (uppercase + trimmed whitespace)
Remove missing values in Make
Filter out aircraft manufacturers with fewer than 50 recorded incidents to ensure robust statistical comparisons

This ensures that conclusions are not biased by rare or underrepresented aircraft makes.

In [36]:
df["Make"] = df["Make"].str.upper().str.strip()

In [37]:
df = df.dropna(subset=["Make"])

In [38]:
make_counts = df["Make"].value_counts()
make_counts.head(10)

Make
CESSNA      24115
PIPER       12414
BEECH        3356
BELL         1686
BOEING       1227
MOONEY       1042
ROBINSON     1035
GRUMMAN       830
BELLANCA      776
HUGHES        716
Name: count, dtype: int64

In [39]:
valid_makes = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(valid_makes)]

In [40]:
df["Make"].value_counts().head(10)

Make
CESSNA      24115
PIPER       12414
BEECH        3356
BELL         1686
BOEING       1227
MOONEY       1042
ROBINSON     1035
GRUMMAN       830
BELLANCA      776
HUGHES        716
Name: count, dtype: int64

Aircraft manufacturers with fewer than 50 recorded incidents were excluded from the analysis to ensure statistically meaningful comparisons.
This threshold reduces noise from rare manufacturers and ensures that safety metrics such as fatality and destruction rates are based on sufficient sample sizes.
The Make column was also standardized to ensure consistency in grouping across the dataset.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

The dataset contains missing values and inconsistencies in the Make and Model columns.
These columns are essential for aircraft-level comparison, so they must be cleaned before analysis.

The following steps are applied:

Remove rows with missing Make or Model values
Standardize text formatting (uppercase and trimmed whitespace)
Inspect whether Model names are unique within each Make
If not unique, create a combined identifier for each aircraft type (Make + Model)

In [41]:
df = df.dropna(subset=["Make", "Model"])

In [42]:
df["Make"] = df["Make"].str.upper().str.strip()
df["Model"] = df["Model"].str.upper().str.strip()

In [43]:
make_model_counts = df.groupby(["Make", "Model"]).size().reset_index(name="Count")
make_model_counts.head()

,Make,Model,Count
0,AERO COMMANDER,100,33
1,AERO COMMANDER,S2R,58
2,AERONCA,11AC,81
3,AERONCA,15AC,26
4,AERONCA,7AC,271


In [44]:
df.groupby("Make")["Model"].nunique().sort_values(ascending=False).head(10)

Make
CESSNA         162
PIPER          102
BEECH           60
BELL            21
BOEING          17
GRUMMAN         10
ROBINSON        10
MOONEY           9
BELLANCA         9
AIR TRACTOR      8
Name: Model, dtype: int64

The analysis shows that aircraft model names are not unique within each manufacturer.
Different manufacturers may use overlapping model naming conventions, and some models may appear under multiple formatting variations.
Therefore, a combined identifier is required to uniquely represent each aircraft type.

In [45]:
df["Aircraft_ID"] = df["Make"] + " - " + df["Model"]

In [46]:
df["Aircraft_ID"].value_counts().head(10)

Aircraft_ID
CESSNA - 152         2366
CESSNA - 172         1753
CESSNA - 172N        1163
PIPER - PA-28-140     932
CESSNA - 150          829
CESSNA - 172M         798
CESSNA - 172P         689
CESSNA - 182          659
CESSNA - 180          621
CESSNA - 150M         585
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [47]:
cols = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

df[cols].head()

,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
0,Reciprocating,UNK,1.0,Personal,Cruise
2,Reciprocating,IMC,1.0,Personal,Cruise
6,Reciprocating,IMC,1.0,Personal,Unknown
7,Reciprocating,VMC,1.0,Personal,Takeoff
10,Reciprocating,VMC,1.0,Personal,Approach


In [48]:
df[cols].info()

<class 'pandas.core.frame.DataFrame'>
Index: 52420 entries, 0 to 88881
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Engine.Type            49483 non-null  object 
 1   Weather.Condition      50312 non-null  object 
 2   Number.of.Engines      49904 non-null  float64
 3   Purpose.of.flight      49849 non-null  object 
 4   Broad.phase.of.flight  40898 non-null  object 
dtypes: float64(1), object(4)
memory usage: 2.4+ MB


In [49]:
df[cols].isna().sum().sort_values(ascending=False)

Broad.phase.of.flight    11522
Engine.Type               2937
Purpose.of.flight         2571
Number.of.Engines         2516
Weather.Condition         2108
dtype: int64

In [50]:
df["Engine.Type"] = df["Engine.Type"].str.upper().str.strip()

In [51]:
df["Weather.Condition"] = df["Weather.Condition"].str.upper().str.strip()
df["Weather.Condition"] = df["Weather.Condition"].fillna("UNKNOWN")

In [52]:
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")

In [53]:
df["Purpose.of.flight"] = df["Purpose.of.flight"].str.upper().str.strip()
df["Purpose.of.flight"] = df["Purpose.of.flight"].fillna("UNKNOWN")

In [54]:
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].str.upper().str.strip()
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].fillna("UNKNOWN")

In [55]:
df[cols].isna().sum()

Engine.Type              2937
Weather.Condition           0
Number.of.Engines        2516
Purpose.of.flight           0
Broad.phase.of.flight       0
dtype: int64

In [56]:
for c in cols:
    print(df[c].value_counts().head(10))

Engine.Type
RECIPROCATING      45799
TURBO SHAFT         1565
TURBO PROP          1008
UNKNOWN              558
TURBO FAN            461
TURBO JET             81
GEARED TURBOFAN       10
UNK                    1
Name: count, dtype: int64
Weather.Condition
VMC        45769
IMC         3894
UNKNOWN     2108
UNK          649
Name: count, dtype: int64
Number.of.Engines
1.0    44858
2.0     4717
0.0      169
3.0      106
4.0       54
Name: count, dtype: int64
Purpose.of.flight
PERSONAL              29274
INSTRUCTIONAL          8214
UNKNOWN                5938
AERIAL APPLICATION     2864
BUSINESS               2612
POSITIONING             789
OTHER WORK USE          704
AERIAL OBSERVATION      522
FERRY                   463
PUBLIC AIRCRAFT         399
Name: count, dtype: int64
Broad.phase.of.flight
UNKNOWN        11882
LANDING        11009
TAKEOFF         8128
CRUISE          6970
MANEUVERING     4931
APPROACH        4179
TAXI            1266
CLIMB           1220
DESCENT         1198
GO-ARO

Operational variables such as engine type, weather conditions, flight purpose, and phase of flight were standardized through text normalization (uppercase formatting and whitespace removal).
Missing values were retained or labeled as “UNKNOWN” where appropriate, as these categories may represent unrecorded or unclassified operational conditions rather than true null values.
These variables are retained for exploratory analysis to assess their relationship with aircraft safety outcomes such as injury severity and aircraft destruction.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [63]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percent": missing_percent
})

missing_summary

,Missing_Count,Missing_Percent
Airport.Code,22786,43.468142
Airport.Name,21010,40.080122
Publication.Date,9165,17.483785
Report.Status,2951,5.629531
Engine.Type,2937,5.602823
Number.of.Engines,2516,4.799695
Fatal_Injury_Rate,641,1.222816
Total_Occupants,641,1.222816
Serious_Injury_Rate,641,1.222816
Severe_Injury_Rate,641,1.222816


Columns with extremely high missingness (>50%) are removed because they do not provide reliable information for statistical analysis.
Medium-missing columns may be retained if they are relevant to safety outcomes.
Derived metrics are preserved even if they have minor missing values, as they are critical to the analysis.

In [61]:
cols_to_drop = [
    "Schedule",
    "Air.carrier",
    "Longitude",
    "Latitude",
    "FAR.Description",
    "Aircraft.Category"
]

In [64]:
df = df.drop(columns=cols_to_drop, errors="ignore")

Several operational and geographic variables were removed due to excessive missingness (>50%), including Schedule, Air carrier, Latitude, Longitude, FAR Description, and Aircraft Category.
These variables were excluded to ensure analytical reliability and prevent bias from incomplete data coverage.
Remaining variables provide sufficient information to analyze aircraft safety, injury severity, and destruction risk.

Some columns identified for removal were already excluded in earlier preprocessing steps.
To ensure robustness and prevent execution errors, the errors="ignore" parameter was used when dropping columns.
This ensures that only existing columns are removed without interrupting the analysis workflow.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [65]:
df.to_csv("data/cleaned_aviation_data.csv", index=False)